# Task 2.1 — Albero di decisione costruito a mano

Costruiamo un albero di decisione sul file `manuale.csv` (14 campioni bilanciati)
**senza usare il fit automatico**: feature e soglie di split le scegliamo noi
calcolando entropia e information gain (algoritmo ID3 per feature continue).

Documentazione completa con teoria e motivazioni: `docs/02.1_albero_decisionale.md`

## 1. Caricamento dei 14 campioni

Carichiamo `manuale.csv` e teniamo solo le colonne utili: le 4 feature e la LABEL.
Gli identificatori (ACTIONID, USERID, TARGETID) e TIMESTAMP non sono caratteristiche
generalizzabili e non vanno usati come feature.

In [1]:
# Importiamo pandas per leggere e manipolare tabelle
import pandas as pd

# Carichiamo manuale.csv: stavolta è un CSV "vero" (separato da virgole),
# quindi non serve sep="\t" come per i .tsv originali
manuale = pd.read_csv("../data/manuale.csv")

# Teniamo solo le colonne che ci servono per l'albero:
# le 4 feature (gli input) e la LABEL (la risposta da prevedere).
# Doppia parentesi quadra perché passiamo una LISTA di nomi di colonne.
df = manuale[["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3", "LABEL"]]

# Mostriamo TUTTE le righe, le stampa tutte:
df

,FEATURE0,FEATURE1,FEATURE2,FEATURE3,LABEL
0,-0.319991,-0.435701,5.116995,2.140347,0
1,-0.319991,-0.435701,10.628228,0.133387,0
2,-0.319991,-0.435701,0.607805,0.735475,1
3,-0.319991,-0.435701,43.194604,0.534779,0
4,-0.319991,-0.435701,2.110868,0.936171,1
5,10.464667,-0.435701,-0.394237,1.738955,0
6,3.724255,-0.435701,-0.394237,1.136867,1
7,-0.319991,-0.435701,1.609847,1.738955,1
8,-0.319991,-0.435701,4.114953,1.337563,0
9,5.072338,-0.435701,-0.394237,2.140347,1


## 2. Funzioni di supporto: entropia e information gain

- Entropia(S) = −p0·log2(p0) − p1·log2(p1)  (con la convenzione 0·log2(0) = 0)
- IG = Entropia(padre) − media pesata delle entropie dei due figli

Scriviamo due piccole funzioni per non ripetere i calcoli.

In [2]:
# math.log2 = logaritmo in base 2.
from math import log2

def entropia(labels):
    """Calcola l'entropia di un gruppo di campioni, date le loro label (0/1)."""
    n = len(labels)              # quanti campioni ci sono nel gruppo
    if n == 0:
        return 0                 # gruppo vuoto: nessun disordine per convenzione

    p1 = sum(labels) / n         # frazione di 1 (le label sono 0/1, la somma conta gli 1)
    p0 = 1 - p1                  # frazione di 0 (il resto)

    e = 0
    # convenzione: 0 * log2(0) = 0, quindi sommiamo un termine solo se p > 0
    # (log2(0) non esiste e darebbe errore)
    if p0 > 0:
        e -= p0 * log2(p0)
    if p1 > 0:
        e -= p1 * log2(p1)
    return e


def info_gain(labels_padre, labels_sx, labels_dx):
    """Information Gain di uno split che divide il padre nei due rami sx e dx."""
    n = len(labels_padre)
    peso_sx = len(labels_sx) / n   # frazione di campioni finita nel ramo sinistro
    peso_dx = len(labels_dx) / n   # frazione nel ramo destro

    # IG = entropia prima dello split - media pesata delle entropie dopo
    return entropia(labels_padre) - (peso_sx * entropia(labels_sx) +
                                     peso_dx * entropia(labels_dx))


# --- mini-test per verificare che funzionino ---
print(entropia([0, 0, 0]))        # atteso: 0.0   (gruppo puro)
print(entropia([0, 1]))           # atteso: 1.0   (50/50, massimo disordine)
print(entropia([1, 1, 1, 0]))     # atteso: ~0.811 (3 contro 1)

0.0
1.0
0.8112781244591328


## 3. Entropia della radice

Il nodo radice contiene tutti e 14 i campioni (7 dropout, 7 no):
l'entropia attesa è esattamente **1.0** (massima incertezza).

In [3]:
# L'entropia della radice: passiamo alla funzione la colonna LABEL intera
# (tutti e 14 i campioni). La convertiamo in lista per semplicità.
e_radice = entropia(list(df["LABEL"]))
print("Entropia della radice:", e_radice)

Entropia della radice: 1.0


## 4. Ricerca dello split migliore per la radice

Per ogni feature:
1. ordiniamo i 14 campioni per quella feature;
2. le soglie candidate sono i punti medi tra valori consecutivi di classe diversa;
3. per ogni candidata calcoliamo l'IG e teniamo la migliore.

Alla fine confrontiamo le 4 migliori (una per feature) e scegliamo la radice.

Almeno un calcolo di IG va rifatto **a mano su carta** e riportato nella documentazione.

In [4]:
# Per ogni feature stampiamo i valori ordinati con la label affiancata:
# serve a vedere DOVE la label cambia scorrendo i valori, cioè dove
# hanno senso le soglie candidate (come nell'esempio ore_studio/esito)
for feat in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    print(feat)
    # ordiniamo le 14 righe per il valore di questa feature
    ordinato = df.sort_values(feat)
    # stampiamo coppie (valore, label) in ordine crescente
    for _, riga in ordinato.iterrows():
        print(f"  {riga[feat]:8.4f}   label={int(riga['LABEL'])}")

def migliori_split(dati, feature):
    """Prova tutte le soglie candidate di una feature e restituisce
    la lista (soglia, IG) ordinata dal gain più alto."""
    ordinato = dati.sort_values(feature).reset_index(drop=True)
    valori = list(ordinato[feature])
    labels = list(ordinato["LABEL"])
    risultati = []

    # scorriamo le coppie di campioni consecutivi
    for i in range(len(valori) - 1):
        # soglia candidata SOLO dove la label cambia (altrove l'IG non migliora)
        # e solo se i due valori sono diversi (con valori uguali non si può tagliare)
        if labels[i] != labels[i + 1] and valori[i] != valori[i + 1]:
            soglia = (valori[i] + valori[i + 1]) / 2   # punto medio

            # dividiamo le label nei due rami secondo la domanda "feature <= soglia?"
            sx = [l for v, l in zip(valori, labels) if v <= soglia]  # ramo "sì"
            dx = [l for v, l in zip(valori, labels) if v > soglia]   # ramo "no"

            risultati.append((soglia, info_gain(labels, sx, dx)))

    # ordiniamo dal gain più alto al più basso
    return sorted(risultati, key=lambda x: x[1], reverse=True)


# Proviamo tutte e 4 le feature e stampiamo le loro candidate migliori
for feat in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    candidate = migliori_split(df, feat)
    print("=" * 60)
    print(feat, "-", len(candidate), "soglie candidate")
    for soglia, ig in candidate[:3]:   # mostriamo solo le 3 migliori per feature
        print(f"   soglia {soglia:8.4f}  ->  IG = {ig:.4f}")

FEATURE0
   -0.3200   label=0
   -0.3200   label=0
   -0.3200   label=1
   -0.3200   label=0
   -0.3200   label=1
   -0.3200   label=1
   -0.3200   label=0
   -0.3200   label=0
   -0.3200   label=1
   -0.3200   label=1
    3.7243   label=0
    3.7243   label=1
    5.0723   label=1
   10.4647   label=0
FEATURE1
   -0.4357   label=0
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=1
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=0
   -0.4357   label=0
   -0.4357   label=1
    2.1087   label=1
FEATURE2
   -0.3942   label=1
   -0.3942   label=0
   -0.3942   label=0
   -0.3942   label=1
   -0.3942   label=1
    0.6078   label=0
    0.6078   label=1
    0.6078   label=1
    1.6098   label=1
    2.1109   label=1
    4.1150   label=0
    5.1170   label=0
   10.6282   label=0
   43.1946   label=0
FEATURE3
   -0.0673   label=1
    0.1334   label=0
    0.5348   label=1
    0.5348   label=

## 5. Costruzione dell'albero (profondità max 2-3)

Scelta la radice, guardiamo i due rami: se uno è impuro ripetiamo la ricerca
dello split **solo sui campioni di quel ramo**. Ci fermiamo quando i rami sono
puri o alla profondità massima, assegnando alle foglie la classe di maggioranza.

L'albero finale, scritto come regole esplicite:
```
SE FEATURE2 > 3.1129:                    → classe 0   (foglia pura, 4 campioni)
ALTRIMENTI:
    SE FEATURE0 > 7.7685:                → classe 0   (foglia pura, 1 campione)
    ALTRIMENTI:
        SE FEATURE3 > 4.9501:            → classe 0   (foglia pura, 1 campione)
        ALTRIMENTI:                      → classe 1   (foglia di maggioranza: 7 sì, 1 no
                                                       — il "gemello" indivisibile)
```


In [5]:
# Il ramo destro della radice (FEATURE2 > 3.1129) è puro: foglia "classe 0".
# Il ramo sinistro (FEATURE2 <= 3.1129) contiene 10 campioni (7 uni, 3 zeri):
# come da algoritmo ricorsivo, ripetiamo la ricerca dello split SOLO su di loro
ramo_sx = df[df["FEATURE2"] <= 3.1129]

print("Campioni nel ramo sinistro:", len(ramo_sx))
print("Entropia del ramo:", entropia(list(ramo_sx["LABEL"])))
print()

# stessa identica procedura usata per la radice, ma sul sottoinsieme
for feat in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    candidate = migliori_split(ramo_sx, feat)
    print("=" * 60)
    print(feat, "-", len(candidate), "soglie candidate")
    for soglia, ig in candidate[:3]:
        print(f"   soglia {soglia:8.4f}  ->  IG = {ig:.4f}")


Campioni nel ramo sinistro: 10
Entropia del ramo: 0.8812908992306927

FEATURE0 - 2 soglie candidate
   soglia   7.7685  ->  IG = 0.1935
   soglia   1.7021  ->  IG = 0.0913
FEATURE1 - 0 soglie candidate
FEATURE2 - 1 soglie candidate
   soglia   0.1068  ->  IG = 0.0349
FEATURE3 - 1 soglie candidate
   soglia   4.9501  ->  IG = 0.1935


In [6]:
# Terzo livello: dentro il ramo FEATURE2 <= 3.1129 e FEATURE0 <= 7.7685
# restano 9 campioni (7 uni, 2 zeri): cerchiamo l'ultimo split
ramo_sx2 = ramo_sx[ramo_sx["FEATURE0"] <= 7.7685]

print("Campioni:", len(ramo_sx2), "- Entropia:", round(entropia(list(ramo_sx2["LABEL"])), 4))
for feat in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    candidate = migliori_split(ramo_sx2, feat)
    for soglia, ig in candidate[:1]:
        print(f"{feat}: soglia {soglia:8.4f} -> IG = {ig:.4f}")

Campioni: 9 - Entropia: 0.7642
FEATURE0: soglia   1.7021 -> IG = 0.0248
FEATURE3: soglia   4.9501 -> IG = 0.2810


## 6. Implementazione dell'albero in Python

L'albero è già "addestrato" (le soglie le abbiamo scelte noi): la sua
implementazione è una semplice catena di if/else.

In [7]:
def predici(riga):
    """Applica l'albero di decisione costruito a mano a una riga del dataset.
    Restituisce la classe predetta: 1 = dropout, 0 = no.
    Le soglie NON vengono da nessuna libreria: sono quelle scelte da noi
    massimizzando l'information gain (vedi celle precedenti)."""
    if riga["FEATURE2"] > 3.1129:
        return 0            # foglia pura (4 campioni)
    if riga["FEATURE0"] > 7.7685:
        return 0            # foglia pura (1 campione)
    if riga["FEATURE3"] > 4.9501:
        return 0            # foglia pura (1 campione)
    return 1                # foglia di maggioranza (7 sì / 1 no)


# Applichiamo l'albero a tutte le 14 righe:
# axis=1 dice ad apply di passare a predici una RIGA alla volta
df["PREDETTA"] = df.apply(predici, axis=1)

# Confronto visivo: label vera contro predetta
df[["FEATURE0", "FEATURE2", "FEATURE3", "LABEL", "PREDETTA"]]

,FEATURE0,FEATURE2,FEATURE3,LABEL,PREDETTA
0,-0.319991,5.116995,2.140347,0,0
1,-0.319991,10.628228,0.133387,0,0
2,-0.319991,0.607805,0.735475,1,1
3,-0.319991,43.194604,0.534779,0,0
4,-0.319991,2.110868,0.936171,1,1
5,10.464667,-0.394237,1.738955,0,0
6,3.724255,-0.394237,1.136867,1,1
7,-0.319991,1.609847,1.738955,1,1
8,-0.319991,4.114953,1.337563,0,0
9,5.072338,-0.394237,2.140347,1,1


## 7. Valutazione sulle 14 osservazioni

Confrontiamo PREDETTA con LABEL:
- **Confusion matrix**: TP, TN, FP, FN
- **Accuracy** = (TP+TN)/14, **Precision** = TP/(TP+FP), **Recall** = TP/(TP+FN),
  **F1** = 2·P·R/(P+R)

Nota: valutare sugli stessi campioni usati per costruire l'albero dà una stima
ottimistica — lo discutiamo nell'analisi critica e lo verificheremo allo Step 4.

In [8]:
# Confronto tra label vera e predetta: i 4 casi della confusion matrix.
# (df.LABEL==1) & (df.PREDETTA==1) produce una colonna di True/False;
# .sum() conta i True (True vale 1 in Python)
TP = ((df["LABEL"] == 1) & (df["PREDETTA"] == 1)).sum()  # dropout riconosciuti
TN = ((df["LABEL"] == 0) & (df["PREDETTA"] == 0)).sum()  # "continua" riconosciuti
FP = ((df["LABEL"] == 0) & (df["PREDETTA"] == 1)).sum()  # falsi allarmi
FN = ((df["LABEL"] == 1) & (df["PREDETTA"] == 0)).sum()  # dropout mancati

print("Confusion matrix:")
print(f"  TP={TP}  FN={FN}")
print(f"  FP={FP}  TN={TN}")
print()

# Le quattro metriche, dalla definizione
accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

Confusion matrix:
  TP=7  FN=0
  FP=1  TN=6

Accuracy : 0.9286
Precision: 0.8750
Recall   : 1.0000
F1-score : 0.9333


## 8. Analisi critica

L'albero costruito a mano classifica correttamente 13 osservazioni su 14
(accuracy 92.9%, F1 0.93).

- **La valutazione è ottimistica.** L'albero è stato costruito e valutato sulle
  stesse 14 osservazioni: sta misurando quanto si adatta a questi campioni, non
  quanto funzionerebbe su dati nuovi. La verifica vera è quella del Task 4.
- **L'unico errore è inevitabile.** Il campione sbagliato (riga 10) ha le stesse
  identiche feature della riga 6 ma classe opposta: due osservazioni
  indistinguibili con esiti diversi. Nessun classificatore basato su queste 4
  feature può azzeccarle entrambe, quindi 13/14 è il massimo raggiungibile su
  questo file. È un limite dei dati, non dell'albero.
- **Le foglie da un solo campione sono un campanello d'allarme.** Il secondo e il
  terzo split isolano un singolo campione ciascuno: con così pochi dati non
  possiamo sapere se le soglie 7.7685 e 4.9501 descrivono una regolarità vera o
  solo il caso di questi specifici campioni (rischio di overfitting).
- **L'albero è instabile.** Con 14 campioni basterebbe sostituirne uno per
  cambiare le soglie o addirittura la feature di radice. Anche il pareggio di
  information gain al secondo split (FEATURE0 e FEATURE3 a 0.1935) mostra che
  la struttura scelta non è l'unica possibile.

